# Bronze Layer Data Ingestion

Reading and ingesting data from the Excel file into the bronze layer of the catalog.

In [0]:
# First, let's discover all sheets in the Excel file
excel_path = "/Volumes/workspace/default/data_modeling/Data model.xlsx"

# List all sheets
sheets_df = spark.sql(f"""
    SELECT * FROM read_files(
        '{excel_path}',
        format => 'excel',
        operation => 'listSheets'
    )
""")

print("Available sheets in the Excel file:")
display(sheets_df)

Available sheets in the Excel file:


sheetIndex,sheetName
0,factSalesTable
1,dimCustomerTable
2,dimDateTable
3,dimRegionTable
4,dimProductTable
5,dimProductSubcategoryTable
6,dimProductCategoryTable


In [0]:
# Get the list of sheet names
sheet_names = [row.sheetName for row in sheets_df.collect()]

print(f"Found {len(sheet_names)} sheets: {sheet_names}\n")

# Read and display each sheet
for sheet_name in sheet_names:
    print(f"\n{'='*80}")
    print(f"Sheet: {sheet_name}")
    print(f"{'='*80}\n")
    
    # Read the sheet - first read to get column names from first row
    df_raw = spark.read.format("excel") \
        .option("header", "false") \
        .option("dataAddress", sheet_name) \
        .load(excel_path)
    
    # Get column names from the first row and sanitize them (replace spaces with underscores)
    first_row = df_raw.first()
    column_names = [first_row[i].replace(" ", "_") for i in range(len(df_raw.columns))]
    
    # Skip the header row and rename columns
    df = df_raw.filter(df_raw["_c0"] != first_row[0]).toDF(*column_names)
    
    # Display sheet info
    print(f"Rows: {df.count()}")
    print(f"Columns: {len(df.columns)}")
    print(f"\nSchema:")
    df.printSchema()
    
    print(f"\nSample data (first 10 rows):")
    display(df.limit(10))

Found 7 sheets: ['factSalesTable', 'dimCustomerTable', 'dimDateTable', 'dimRegionTable', 'dimProductTable', 'dimProductSubcategoryTable', 'dimProductCategoryTable']


Sheet: factSalesTable

Rows: 114390
Columns: 8

Schema:
root
 |-- ProductKey: string (nullable = true)
 |-- OrderDateKey: string (nullable = true)
 |-- CustomerKey: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- OrderNumber: string (nullable = true)
 |-- OrderQuantity: string (nullable = true)
 |-- List_Price: string (nullable = true)
 |-- Product_Cost: string (nullable = true)


Sample data (first 10 rows):


ProductKey,OrderDateKey,CustomerKey,Gender,OrderNumber,OrderQuantity,List_Price,Product_Cost
344,20050722,11000,null,20061722,22,3399.99,1912.1543999999999
353,20070722,11000,null,20081722,22,2319.9899999999998,1265.6195
485,20070722,11000,null,20081722,22,21.98,8.2204999999999995
530,20071104,11000,null,20082104,4,4.99,1.8663000000000001
214,20071104,11000,null,20082104,4,34.99,13.0863
488,20071104,11000,null,20082104,4,53.99,41.572299999999998
573,20071104,11000,null,20082104,4,2384.0700000000002,1481.9378999999999
541,20071104,11000,null,20082104,4,28.99,10.8423
350,20050718,11001,null,20061719,18,3374.99,1898.0944
477,20070720,11001,null,20081721,20,4.99,1.8663000000000001



Sheet: dimCustomerTable

Rows: 9999
Columns: 5

Schema:
root
 |-- GeographyKey: string (nullable = true)
 |-- MaritalStatus: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- CustomerKey: string (nullable = true)
 |-- CustomerName: string (nullable = true)


Sample data (first 10 rows):


GeographyKey,MaritalStatus,Gender,CustomerKey,CustomerName
612,M,M,11254,Customer1
612,M,M,11255,Customer2
612,M,M,11282,Customer3
612,M,M,11321,Customer4
612,M,M,11633,Customer5
612,M,M,11799,Customer6
612,M,M,12092,Customer7
612,M,M,12436,Customer8
612,M,M,12456,Customer9
612,M,M,12906,Customer10



Sheet: dimDateTable

Rows: 1188
Columns: 4

Schema:
root
 |-- DateKey: string (nullable = true)
 |-- FullDateAlternateKey: string (nullable = true)
 |-- EnglishMonthName: string (nullable = true)
 |-- CalendarYear: string (nullable = true)


Sample data (first 10 rows):


...

ProductKey,ProductSubcategoryKey,Color,StandardCost,ListPrice,Product_Name
212,31,Red,12.027799999999999,33.644199999999998,Product 1
213,31,Red,13.8782,33.644199999999998,Product 2
214,31,Red,13.0863,34.99,Product 3
218,23,White,3.3963000000000001,9.5,Product 4
219,23,White,3.3963000000000001,9.5,Product 5
220,31,Blue,12.027799999999999,33.644199999999998,Product 6
221,31,Blue,13.8782,33.644199999999998,Product 7
222,31,Blue,13.0863,34.99,Product 8
223,19,Multi,5.7051999999999996,8.6441999999999997,Product 9
224,19,Multi,5.2297000000000002,8.6441999999999997,Product 10



Sheet: dimProductSubcategoryTable

Rows: 40
Columns: 3

Schema:
root
 |-- ProductSubcategoryKey: string (nullable = true)
 |-- ProductSubcategoryName: string (nullable = true)
 |-- ProductCategoryKey: string (nullable = true)


Sample data (first 10 rows):


ProductSubcategoryKey,ProductSubcategoryName,ProductCategoryKey
1,Mountain Bikes,1
2,Road Bikes,1
3,Touring Bikes,1
4,Handlebars,2
5,Bottom Brackets,2
6,Brakes,2
7,Chains,2
8,Cranksets,2
9,Derailleurs,2
10,Forks,2



Sheet: dimProductCategoryTable

Rows: 5
Columns: 2

Schema:
root
 |-- ProductCategoryKey: string (nullable = true)
 |-- ProductCategoryName: string (nullable = true)


Sample data (first 10 rows):


ProductCategoryKey,ProductCategoryName
1,Bikes
2,Components
3,Clothing
4,Accessories
5,Food


In [0]:
# Define the bronze catalog and schema
bronze_catalog = "workspace"
bronze_schema = "bronze"

# Ensure the bronze schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_catalog}.{bronze_schema}")

print(f"Saving sheets to {bronze_catalog}.{bronze_schema}\n")

# Save each sheet as a table in the bronze layer
for sheet_name in sheet_names:
    # Create a valid table name from sheet name (replace spaces and special chars with underscores)
    table_name = sheet_name.lower().replace(' ', '_').replace('-', '_').replace('(', '').replace(')', '')
    full_table_name = f"{bronze_catalog}.{bronze_schema}.{table_name}"
    
    print(f"Processing sheet: {sheet_name} -> {full_table_name}")
    
    # Read the sheet - first read to get column names from first row
    df_raw = spark.read.format("excel") \
        .option("header", "false") \
        .option("dataAddress", sheet_name) \
        .load(excel_path)
    
    # Get column names from the first row and sanitize them (replace spaces with underscores)
    first_row = df_raw.first()
    column_names = [first_row[i].replace(" ", "_") for i in range(len(df_raw.columns))]
    
    # Skip the header row and rename columns
    df = df_raw.filter(df_raw["_c0"] != first_row[0]).toDF(*column_names)
    
    # Write to bronze layer as a Delta table (overwrite schema to update column names)
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table_name)
    
    row_count = spark.table(full_table_name).count()
    print(f"✓ Saved {row_count} rows to {full_table_name}\n")

print(f"\n{'='*80}")
print("All sheets successfully saved to bronze layer!")
print(f"{'='*80}")

Saving sheets to workspace.bronze

Processing sheet: factSalesTable -> workspace.bronze.factsalestable
✓ Saved 114390 rows to workspace.bronze.factsalestable

Processing sheet: dimCustomerTable -> workspace.bronze.dimcustomertable
✓ Saved 9999 rows to workspace.bronze.dimcustomertable

Processing sheet: dimDateTable -> workspace.bronze.dimdatetable
✓ Saved 1188 rows to workspace.bronze.dimdatetable

Processing sheet: dimRegionTable -> workspace.bronze.dimregiontable
✓ Saved 655 rows to workspace.bronze.dimregiontable

Processing sheet: dimProductTable -> workspace.bronze.dimproducttable
✓ Saved 400 rows to workspace.bronze.dimproducttable

Processing sheet: dimProductSubcategoryTable -> workspace.bronze.dimproductsubcategorytable
✓ Saved 40 rows to workspace.bronze.dimproductsubcategorytable

Processing sheet: dimProductCategoryTable -> workspace.bronze.dimproductcategorytable
✓ Saved 5 rows to workspace.bronze.dimproductcategorytable


All sheets successfully saved to bronze layer!
